# Out-of-Core Pipeline – Análisis de rendimiento (CU\_OOC)

Este notebook **descubre automáticamente** todas las escenas bajo `benchmark/nsight_gr_results/CU_OOC/` (cada subcarpeta con al menos un `*_analysis.yaml`), extrae métricas de los YAML exportados por Nsight Graphics y, si existen, cruza con los CSV de preprocesamiento y stats batch.

**Estructura esperada por escena:**
- Carpeta: `CU_OOC/<id_escena>/YAML/range{anillo}_{posición}_analysis.yaml` (también se aceptan YAML en la raíz de `<id_escena>` si no hay subcarpeta `YAML`).
- Cualquier archivo cuyo nombre coincida con `range\d+_\d+_analysis.yaml` se carga; no hace falta listar escenas a mano.

**CSVs opcionales:**
- Preprocesamiento: `media/csv/cuda_outofcore/ooc_preprocess_metrics.csv`
- Stats por batch: `media/csv/cuda_outofcore/ooc_stats_batch.csv`

**Convención de nombres:** `range{R}_{P}` — anillo R, posición P (p. ej. 5×3 capturas).

In [ ]:
import os, re, glob, yaml, json, warnings
from pathlib import Path
from collections import defaultdict
from statistics import mean

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

## 0. Configuración de rutas

In [ ]:
PROJECT_ROOT = Path(os.path.abspath('')).parent.parent
CU_OOC_ROOT = PROJECT_ROOT / 'benchmark' / 'nsight_gr_results' / 'CU_OOC'
RANGE_YAML_RE = re.compile(r'^range(\d+)_(\d+)_analysis\.yaml$', re.IGNORECASE)

# Etiquetas legibles para carpetas conocidas (el resto: "CU_OOC / <nombre>")
SCENE_LABEL_HINTS = {
    '8wql': 'Molécula 8wql (LOADED_SCENE)',
    'g10m': 'PACKAGE_SCENE ~10M esferas',
    'g50m': 'PACKAGE_SCENE ~50M esferas',
    'g100m': 'PACKAGE_SCENE ~100M esferas',
    'g500m': 'PACKAGE_SCENE ~500M esferas',
}


def _scene_sort_key(name: str):
    m = re.match(r'^g(\d+)m$', name, re.IGNORECASE)
    if m:
        return (0, int(m.group(1)))
    return (1, name.lower())


def _label_for_folder(folder_name: str) -> str:
    return SCENE_LABEL_HINTS.get(folder_name, f'CU_OOC / {folder_name}')


def discover_cu_ooc_scenes(cu_ooc: Path) -> dict:
    """
    Recorre CU_OOC: cada subdirectorio con al menos un range*_analysis.yaml.
    Prioriza <escena>/YAML/; si no existe o está vacío, usa la raíz de la escena.
    """
    found = {}
    if not cu_ooc.is_dir():
        return found
    for child in sorted(cu_ooc.iterdir(), key=lambda p: p.name.lower()):
        if not child.is_dir():
            continue
        yaml_dir = child / 'YAML'
        if not yaml_dir.is_dir():
            yaml_dir = child
        elif not any(yaml_dir.glob('*_analysis.yaml')):
            yaml_dir = child
        yfiles = sorted(yaml_dir.glob('*_analysis.yaml'))
        valid = [fp for fp in yfiles if RANGE_YAML_RE.match(fp.name)]
        if not valid:
            continue
        found[child.name] = {
            'label': _label_for_folder(child.name),
            'yaml_dir': yaml_dir,
            'yaml_files': valid,
        }
    return {k: found[k] for k in sorted(found.keys(), key=_scene_sort_key)}


SCENES = discover_cu_ooc_scenes(CU_OOC_ROOT)

CSV_BASE = PROJECT_ROOT / 'out' / 'build' / 'vs-release' / 'media' / 'csv' / 'cuda_outofcore'
CSV_BASE_FALLBACK = PROJECT_ROOT / 'media' / 'csv' / 'cuda_outofcore'

def _csv_dir():
    if CSV_BASE.is_dir():
        return CSV_BASE
    return CSV_BASE_FALLBACK

PREPROCESS_CSV = _csv_dir() / 'ooc_preprocess_metrics.csv'

BATCH_CSV_RE_FILE = re.compile(r'^ooc_stats_batch_(.+)\.csv$', re.IGNORECASE)
def discover_batch_csvs():
    d = _csv_dir()
    if not d.is_dir():
        return {}
    out = {}
    for f in sorted(d.iterdir()):
        m = BATCH_CSV_RE_FILE.match(f.name)
        if m:
            out[m.group(1)] = f
    return out

BATCH_CSVS = discover_batch_csvs()

OOC_PIPELINE_RANGES = [
    'Screen Clear',
    'Octree BFS Frustum Culling',
    'Compute Block Depth+Area',
    'Thrust Sort (Depth)',
    'Occlusion Culling',
    'Request Generation',
    'Build Active Atom List',
    'Sphere Raster OOC',
    'HiZ Downsample',
    'cub::DeviceRadixSort',
    'Blit Framebuffer',
    'Swap Window',
]

KEY_METRIC_IDS = {
    'gr_active_pct':   'gr__cycles_active.avg.pct_of_peak_sustained_elapsed',
    'gr_idle_pct':     'oracle.gr__cycles_idle.pct',
    'l1tex_hit_pct':   'l1tex__t_sector_hit_rate.pct',
    'pcie_throughput':  'pcie__throughput.avg.pct_of_peak_sustained_elapsed',
    'sm_issue_active':  'sm__inst_executed_realtime.avg.pct_of_peak_sustained_elapsed',
    'sm_pipe_alu':      'sm__inst_executed_pipe_alu_realtime.avg.pct_of_peak_sustained_elapsed',
    'warp_occ_pct':     'sm__warps_active.avg.pct_of_peak_sustained_elapsed',
    'thread_active_pct': 'sm__average_thread_inst_executed_pred_on_per_inst_executed_realtime.pct',
    'stall_short_scoreboard': 'smsp__warps_issue_stalled_short_scoreboard.avg.pct_of_peak_sustained_elapsed',
    'stall_drain':      'smsp__warps_issue_stalled_drain.avg.pct_of_peak_sustained_elapsed',
    'stall_wait':       'smsp__warps_issue_stalled_wait.avg.pct_of_peak_sustained_elapsed',
    'sm_idle_pct':      'tpc__warps_inactive_sm_idle.avg.pct_of_peak_sustained_elapsed',
}

print('Proyecto:', PROJECT_ROOT)
print('Raíz CU_OOC:', CU_OOC_ROOT, '—', 'OK' if CU_OOC_ROOT.is_dir() else 'NO ENCONTRADA')
print(f'Escenas detectadas: {len(SCENES)}')
for k, v in SCENES.items():
    rings = {int(RANGE_YAML_RE.match(f.name).group(1)) for f in v['yaml_files']}
    poss = {int(RANGE_YAML_RE.match(f.name).group(2)) for f in v['yaml_files']}
    print(f"  [{k}] {v['label']}")
    print(f"       YAML: {v['yaml_dir']}  ({len(v['yaml_files'])} archivos, anillos {min(rings)}–{max(rings)}, posiciones {sorted(poss)})")
print(f"  Preprocess CSV: {PREPROCESS_CSV} -- {'OK' if PREPROCESS_CSV.exists() else 'NO ENCONTRADO'}")
print(f"  Batch CSVs detectados: {len(BATCH_CSVS)}")
for bk, bp in BATCH_CSVS.items():
    print(f"    [{bk}] {bp}")

## 1. Parsing de YAML (Nsight Trace Analysis)

In [ ]:
def parse_yaml_trace(filepath):
    """Parse a Nsight GPU Trace analysis YAML, returning a list of dicts per range entry."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    if data is None:
        return []
    return data


def extract_metrics_from_range_entry(entry):
    """From one YAML range entry, extract all metric Id→Value pairs."""
    metrics = {}
    for rule in entry.get('Rules', []) or []:
        for m in rule.get('Metrics', []) or []:
            mid = m.get('Id', '')
            val = m.get('Value')
            if mid and val is not None:
                try:
                    metrics[mid] = float(val)
                except (ValueError, TypeError):
                    pass
    metrics['RelativeFrameDuration'] = entry.get('RelativeFrameDuration', 0.0)
    return metrics


def aggregate_yaml_file(filepath, target_ranges=None):
    """
    Parse one YAML file with multiple frames of Nsight data.
    Each range name repeats once per frame. We average metrics across frames
    for each unique range name.
    Returns dict: range_name -> {metric_id: avg_value}
    """
    entries = parse_yaml_trace(filepath)
    buckets = defaultdict(list)  # range_name -> [metrics_dict, ...]
    for e in entries:
        rname = e.get('Range', '')
        if target_ranges and rname not in target_ranges:
            continue
        buckets[rname].append(extract_metrics_from_range_entry(e))

    result = {}
    for rname, mlist in buckets.items():
        all_keys = set()
        for m in mlist:
            all_keys.update(m.keys())
        avg = {}
        for k in all_keys:
            vals = [m[k] for m in mlist if k in m]
            if vals:
                avg[k] = np.mean(vals)
        result[rname] = avg
    return result


print('Funciones de parsing listas.')

## 2. Carga de todos los YAML por escena

In [ ]:
all_scene_data = {}  # scene_key -> {sample_label -> {range_name -> {metric_id: val}}}

for scene_key, scene_cfg in SCENES.items():
    scene_data = {}
    for fpath in scene_cfg['yaml_files']:
        m = RANGE_YAML_RE.match(fpath.name)
        if not m:
            continue
        ring, pos = int(m.group(1)), int(m.group(2))
        label = f'range{ring}_{pos}'
        try:
            agg = aggregate_yaml_file(str(fpath), set(OOC_PIPELINE_RANGES))
        except Exception as e:
            print(f'[ERROR] {scene_key} / {fpath.name}: {e}')
            continue
        scene_data[label] = agg

    all_scene_data[scene_key] = scene_data
    print(f"[OK] {scene_key}: {len(scene_data)} muestras YAML extraídas")

if not all_scene_data:
    print('⚠ No se cargaron datos. Añade carpetas con range*_analysis.yaml bajo', CU_OOC_ROOT)

### 2b. Tabla resumen de extracción (todas las escenas CU\_OOC)

In [ ]:
rows_sum = []
for sk, sd in all_scene_data.items():
    cfg = SCENES[sk]
    rings = [int(RANGE_YAML_RE.match(f.name).group(1)) for f in cfg['yaml_files']]
    poss = [int(RANGE_YAML_RE.match(f.name).group(2)) for f in cfg['yaml_files']]
    rows_sum.append({
        'escena_id': sk,
        'etiqueta': cfg['label'],
        'n_yaml': len(cfg['yaml_files']),
        'anillo_min': min(rings) if rings else None,
        'anillo_max': max(rings) if rings else None,
        'posiciones': sorted(set(poss)),
        'carpeta_yaml': str(cfg['yaml_dir']),
    })
df_extract = pd.DataFrame(rows_sum)
if not df_extract.empty:
    display(df_extract)
else:
    print('Sin filas: no hay escenas con YAML válidos.')

## 3. DataFrame consolidado de métricas GPU por rango del pipeline

In [ ]:
def build_metrics_dataframe(scene_data, key_metrics=KEY_METRIC_IDS):
    """
    Build a tidy DataFrame: one row per (sample, pipeline_range), columns for each key metric.
    """
    rows = []
    for sample_label, ranges in scene_data.items():
        parts = sample_label.replace('range', '').split('_')
        ring = int(parts[0])
        pos = int(parts[1])
        for rname, mdict in ranges.items():
            row = {
                'sample': sample_label,
                'ring': ring,
                'position': pos,
                'pipeline_range': rname,
                'rel_frame_duration': mdict.get('RelativeFrameDuration', 0.0),
            }
            for friendly, mid in key_metrics.items():
                row[friendly] = mdict.get(mid, np.nan)
            rows.append(row)
    return pd.DataFrame(rows)


dfs = {}
for scene_key, scene_data in all_scene_data.items():
    df = build_metrics_dataframe(scene_data)
    dfs[scene_key] = df
    print(f"\n=== {SCENES[scene_key]['label']} ===")
    print(f"Filas: {len(df)}, Muestras únicas: {df['sample'].nunique()}, Rangos: {df['pipeline_range'].nunique()}")
    display(df.head(10))

### 3b. Panorama: escenas `g*m` vs tamaño inferido (millones de esferas)

Si las carpetas siguen el patrón `g10m`, `g50m`, etc., se interpreta el número como millones de entidades solo para **gráficos comparativos** entre escenas (no sustituye al `sphere_count` del CSV de preprocess).

In [ ]:
def infer_sphere_millions(scene_id: str):
    m = re.match(r'^g(\d+)m$', scene_id, re.IGNORECASE)
    return int(m.group(1)) if m else None

pkg_scenes = [(k, infer_sphere_millions(k)) for k in dfs if infer_sphere_millions(k) is not None]
if len(pkg_scenes) >= 2:
    target_rng = 'Sphere Raster OOC'
    all_rings = set()
    for sk, _ in pkg_scenes:
        all_rings.update(dfs[sk]['ring'].unique())
    fig, ax = plt.subplots(figsize=(11, 5))
    plotted = False
    for ring_id in sorted(all_rings):
        xs, ys = [], []
        for sk, millions in sorted(pkg_scenes, key=lambda t: t[1]):
            d = dfs[sk]
            sub = d[(d['pipeline_range'] == target_rng) & (d['ring'] == ring_id)]
            if sub.empty:
                continue
            xs.append(millions)
            ys.append(sub['rel_frame_duration'].mean())
        if len(xs) >= 2:
            ax.plot(xs, ys, marker='o', label=f'Anillo {ring_id}')
            plotted = True
    if plotted:
        ax.set_xscale('log')
        ax.set_xlabel('Millones de esferas (inferido del nombre de carpeta g*m, escala log)')
        ax.set_ylabel(f'Fracción media del frame — {target_rng}')
        ax.set_title('Escalas PACKAGE: coste relativo del raster OOC vs tamaño de escena')
        ax.legend(title='Distancia (anillo)')
        plt.tight_layout()
        plt.show()
    else:
        print('No hay suficientes puntos para', target_rng, 'en las escenas g*m.')
else:
    print('Menos de 2 carpetas g*m con datos; se omite el panorama comparativo.')

## 4. Duración relativa por etapa del pipeline (proporción del frame)

In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    pivot = df.pivot_table(
        index='pipeline_range', columns='ring',
        values='rel_frame_duration', aggfunc='mean'
    )
    ordered = [r for r in OOC_PIPELINE_RANGES if r in pivot.index]
    pivot = pivot.loc[ordered]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Stacked bar
    pivot.T.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab20')
    axes[0].set_title(f"{cfg['label']} — Proporción del frame por etapa")
    axes[0].set_xlabel('Anillo (distancia)')
    axes[0].set_ylabel('Fracción del frame')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    axes[0].set_xticklabels([f'Ring {c}' for c in pivot.columns], rotation=0)

    # Heatmap
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1],
                cbar_kws={'label': 'Fracción del frame'})
    axes[1].set_title(f"{cfg['label']} — Heatmap duración relativa")
    axes[1].set_ylabel('Etapa del pipeline')
    axes[1].set_xlabel('Anillo')

    plt.tight_layout()
    plt.show()

## 5. Cuello de botella: etapa más costosa por anillo

In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    grp = df.groupby(['ring', 'pipeline_range'])['rel_frame_duration'].mean().reset_index()
    idx = grp.groupby('ring')['rel_frame_duration'].idxmax()
    bottleneck = grp.loc[idx][['ring', 'pipeline_range', 'rel_frame_duration']]
    bottleneck.columns = ['Anillo', 'Etapa cuello de botella', 'Fracción del frame']
    print(f"\n=== {cfg['label']} — Cuello de botella por distancia ===")
    display(bottleneck.reset_index(drop=True))

## 6. GR Cycles Active & GPU Idle por etapa y anillo

In [ ]:
def plot_metric_by_ring(df, metric_col, title, ylabel, threshold=None, threshold_label=None,
                        palette='viridis', figsize=(16, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    ordered = [r for r in OOC_PIPELINE_RANGES if r in df['pipeline_range'].values]
    sub = df[df['pipeline_range'].isin(ordered)].copy()
    sub['pipeline_range'] = pd.Categorical(sub['pipeline_range'], categories=ordered, ordered=True)

    sns.barplot(data=sub, x='pipeline_range', y=metric_col, hue='ring',
                palette=palette, ax=ax, errorbar='sd')
    if threshold is not None:
        ax.axhline(threshold, ls='--', color='red', lw=1.5,
                   label=threshold_label or f'{threshold}')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Etapa del pipeline')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Anillo')
    plt.tight_layout()
    plt.show()


for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    plot_metric_by_ring(df, 'gr_active_pct',
                        f"{cfg['label']} — GR Cycles Active [%]",
                        'GR Active %', threshold=80, threshold_label='80% (alto uso)')
    plot_metric_by_ring(df, 'gr_idle_pct',
                        f"{cfg['label']} — GPU Idle [%]",
                        'GR Idle %', threshold=10, threshold_label='10% (baja inactividad)')

## 7. L1TEX Hit Rate y PCIe Throughput

In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    plot_metric_by_ring(df, 'l1tex_hit_pct',
                        f"{cfg['label']} — L1TEX Hit Rate [%]",
                        'L1TEX Hit %', threshold=70, threshold_label='70%')
    plot_metric_by_ring(df, 'pcie_throughput',
                        f"{cfg['label']} — PCIe Throughput [%]",
                        'PCIe %', palette='magma')

## 8. Warp Occupancy y Stall Breakdown

In [ ]:
stall_cols = ['stall_short_scoreboard', 'stall_drain', 'stall_wait']

for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]

    plot_metric_by_ring(df, 'warp_occ_pct',
                        f"{cfg['label']} — SM Warp Occupancy [%]",
                        'Warp Occupancy %', palette='crest')

    # Stall stacked
    stall_data = df.groupby(['ring', 'pipeline_range'])[stall_cols].mean().reset_index()
    ordered = [r for r in OOC_PIPELINE_RANGES if r in stall_data['pipeline_range'].values]

    for ring_id in sorted(df['ring'].unique()):
        sub = stall_data[(stall_data['ring'] == ring_id) &
                         stall_data['pipeline_range'].isin(ordered)].copy()
        sub['pipeline_range'] = pd.Categorical(sub['pipeline_range'], categories=ordered, ordered=True)
        sub = sub.sort_values('pipeline_range').set_index('pipeline_range')[stall_cols]

        if sub.dropna(how='all').empty:
            continue

        sub.plot(kind='bar', stacked=True, figsize=(14, 5),
                 color=['#e74c3c', '#3498db', '#2ecc71'])
        plt.title(f"{cfg['label']} — Stall Breakdown, Anillo {ring_id}")
        plt.ylabel('Stall %')
        plt.xlabel('Etapa del pipeline')
        plt.xticks(rotation=45, ha='right')
        plt.legend(['Short Scoreboard', 'Drain', 'Wait'])
        plt.tight_layout()
        plt.show()

## 9. Evolución de métricas clave vs distancia (tendencia por anillo)

Para las etapas más relevantes del pipeline OOC (Sphere Raster, Frustum Culling, Occlusion Culling), trazamos cómo varían las métricas al alejarnos.

In [ ]:
focus_ranges = ['Sphere Raster OOC', 'Octree BFS Frustum Culling',
                'Occlusion Culling', 'Build Active Atom List']
focus_metrics = ['gr_active_pct', 'gr_idle_pct', 'l1tex_hit_pct',
                 'pcie_throughput', 'warp_occ_pct', 'sm_issue_active']

for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    sub = df[df['pipeline_range'].isin(focus_ranges)].copy()
    if sub.empty:
        continue

    for met in focus_metrics:
        if sub[met].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(12, 5))
        for rng in focus_ranges:
            rng_data = sub[sub['pipeline_range'] == rng].groupby('ring')[met].agg(['mean', 'std']).reset_index()
            if rng_data['mean'].dropna().empty:
                continue
            ax.errorbar(rng_data['ring'], rng_data['mean'], yerr=rng_data['std'],
                        marker='o', capsize=4, label=rng)
        ax.set_title(f"{cfg['label']} — {met} vs Distancia")
        ax.set_xlabel('Anillo (1=cerca, 5=lejos)')
        ax.set_ylabel(met)
        ax.legend()
        ax.set_xticks(sorted(df['ring'].unique()))
        plt.tight_layout()
        plt.show()

## 10. Heatmap de correlación entre métricas (por etapa principal)

In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    num_cols = [c for c in KEY_METRIC_IDS.keys() if c in df.columns]

    for rng in focus_ranges:
        sub = df[df['pipeline_range'] == rng][num_cols].dropna(axis=1, how='all')
        if sub.shape[1] < 2:
            continue
        corr = sub.corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, ax=ax, square=True)
        ax.set_title(f"{cfg['label']} — Correlación de métricas: {rng}")
        plt.tight_layout()
        plt.show()

## 11. Preprocesamiento OOC — Tiempos, escalado y memoria

Análisis del CSV `ooc_preprocess_metrics.csv`. Columnas: tiempos de Morton, bloques/fichero, octree; tamaños host/block-file; VRAM estimada.

In [ ]:
if PREPROCESS_CSV.exists():
    df_prep = pd.read_csv(PREPROCESS_CSV)
    df_prep['total_preprocess_ms'] = df_prep[['ms_morton_pipeline','ms_blocks_and_file','ms_octree_build']].sum(axis=1)
    df_prep['atoms_per_ms'] = df_prep['sphere_count'] / df_prep['total_preprocess_ms']
    df_prep['vram_used_MB'] = df_prep['vram_sum_estimated_bytes'] / (1024**2)
    df_prep['block_file_MB'] = df_prep['block_file_bytes'] / (1024**2)
    df_prep['host_MB'] = df_prep['host_structures_bytes'] / (1024**2)
    df_prep['vram_free_MB'] = df_prep['cuda_mem_free_bytes'] / (1024**2)
    df_prep['vram_total_MB'] = df_prep['cuda_mem_total_bytes'] / (1024**2)
    df_prep['vram_pct_used'] = 100.0 * (1.0 - df_prep['cuda_mem_free_bytes'] / df_prep['cuda_mem_total_bytes'])

    prep_grp = df_prep.groupby(['scene_type','sphere_count']).agg(
        n_runs=('total_preprocess_ms','count'),
        avg_ms=('total_preprocess_ms','mean'), std_ms=('total_preprocess_ms','std'),
        min_ms=('total_preprocess_ms','min'), max_ms=('total_preprocess_ms','max'),
        avg_morton=('ms_morton_pipeline','mean'), avg_blocks=('ms_blocks_and_file','mean'),
        avg_octree=('ms_octree_build','mean'),
        avg_throughput=('atoms_per_ms','mean'),
        avg_vram_MB=('vram_used_MB','mean'), avg_vram_pct=('vram_pct_used','mean'),
        avg_block_file_MB=('block_file_MB','mean'), avg_host_MB=('host_MB','mean'),
        block_count=('block_count','first'), octree_nodes=('octree_node_count','first'),
    ).reset_index().sort_values('sphere_count')

    print('=== Preprocesamiento: resumen por escena/tama\u00f1o ===')
    display(prep_grp)

    pkg = prep_grp[prep_grp['scene_type'].str.contains('PACKAGE')].copy()
    if len(pkg) >= 2:
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))

        ax = axes[0,0]
        ax.errorbar(pkg['sphere_count']/1e6, pkg['avg_ms'], yerr=pkg['std_ms'].fillna(0),
                    marker='o', capsize=4)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('Tiempo total (ms)')
        ax.set_title('Tiempo de preprocesamiento vs entidades')

        ax = axes[0,1]
        for col, lbl in [('avg_morton','Morton+sort'), ('avg_blocks','Bloques+fichero'), ('avg_octree','Octree')]:
            ax.plot(pkg['sphere_count']/1e6, pkg[col], marker='o', label=lbl)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('ms')
        ax.set_title('Desglose de fases del preprocesamiento'); ax.legend()

        ax = axes[0,2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_throughput'], marker='s', color='green')
        ax.set_xscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('\u00c1tomos/ms')
        ax.set_title('Throughput de preprocesamiento')

        ax = axes[1,0]
        ax.bar(pkg['sphere_count'].astype(str), pkg['avg_vram_MB'], color='steelblue')
        ax.set_xlabel('Esferas'); ax.set_ylabel('VRAM estimada (MB)')
        ax.set_title('VRAM total estimada'); ax.tick_params(axis='x', rotation=45)

        ax = axes[1,1]
        ax.bar(pkg['sphere_count'].astype(str), pkg['avg_vram_pct'], color='coral')
        ax.axhline(100, ls='--', color='red', lw=1)
        ax.set_xlabel('Esferas'); ax.set_ylabel('% VRAM usada')
        ax.set_title('% GPU VRAM ocupada'); ax.tick_params(axis='x', rotation=45)

        ax = axes[1,2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_block_file_MB'], marker='o', label='Block file')
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_host_MB'], marker='^', label='Host structs')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('MB')
        ax.set_title('Datos en disco y host'); ax.legend()

        plt.tight_layout(); plt.show()
else:
    print(f'\u26a0 CSV de preprocesamiento no encontrado en {PREPROCESS_CSV}')


## 12. Stats Batch — Análisis de runtime por escena

Cada archivo `ooc_stats_batch_<escena>.csv` contiene filas agregadas de N frames.

**Significado de las columnas clave:**
- `avg_numVisible`: promedio de **bloques** visibles tras frustum culling del octree.
- `avg_numFiltered`: promedio de **bloques** que superan el occlusion culling (subconjunto de visibles, es decir, los que sí se procesan).
- `avg_numRequests`: promedio de **bloques** nuevos solicitados al streaming (aún no residentes en el pool GPU).
- `avg_activeCount`: promedio de **átomos efectivamente rasterizados** por el kernel `Sphere Raster OOC`.


In [ ]:
if not BATCH_CSVS:
    print('\u26a0 No se encontraron CSVs de stats batch en', _csv_dir())
else:
    all_batches = []
    for bk, bp in BATCH_CSVS.items():
        tmp = pd.read_csv(bp)
        tmp['escena_id'] = bk
        tmp['label'] = SCENE_LABEL_HINTS.get(bk, bk)
        tmp['avg_fps'] = 1000.0 / tmp['avg_frame_ms'].replace(0, np.nan)
        tmp['min_fps'] = 1000.0 / tmp['max_frame_ms'].replace(0, np.nan)
        tmp['max_fps'] = 1000.0 / tmp['min_frame_ms'].replace(0, np.nan)
        all_batches.append(tmp)
        print(f'[OK] {bk}: {len(tmp)} batches cargados')

    df_batch_all = pd.concat(all_batches, ignore_index=True)

    # --- 12a. Tabla resumen por escena ---
    print('\n=== Resumen por escena ===')
    summary = df_batch_all.groupby('escena_id').agg(
        sphere_count=('sphere_count','first'),
        total_blocks=('total_blocks','first'),
        pool_slots=('pool_slots','first'),
        n_batches=('avg_fps','count'),
        fps_mean=('avg_fps','mean'), fps_std=('avg_fps','std'),
        fps_min=('avg_fps','min'), fps_max=('avg_fps','max'),
        frame_ms_mean=('avg_frame_ms','mean'),
        visible_mean=('avg_numVisible','mean'),
        filtered_mean=('avg_numFiltered','mean'),
        requests_mean=('avg_numRequests','mean'),
        active_mean=('avg_activeCount','mean'),
    ).reset_index()
    display(summary)

    # --- 12b. FPS por escena (boxplot) ---
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))

    ax = axes[0,0]
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_fps', order=order, ax=ax, palette='viridis')
    ax.set_title('Distribuci\u00f3n de FPS por escena'); ax.set_xlabel('Escena'); ax.set_ylabel('FPS')
    ax.tick_params(axis='x', rotation=30)

    ax = axes[0,1]
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_frame_ms', order=order, ax=ax, palette='magma')
    ax.set_title('Tiempo de frame (ms) por escena'); ax.set_xlabel('Escena'); ax.set_ylabel('ms')
    ax.tick_params(axis='x', rotation=30)

    ax = axes[1,0]
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_activeCount', order=order, ax=ax, palette='crest')
    ax.set_title('\u00c1tomos rasterizados por escena (no bloques)'); ax.set_xlabel('Escena'); ax.set_ylabel('\u00c1tomos')
    ax.tick_params(axis='x', rotation=30)

    ax = axes[1,1]
    sns.boxplot(data=df_batch_all, x='escena_id', y='avg_numRequests', order=order, ax=ax, palette='flare')
    ax.set_title('Bloques solicitados a streaming por escena'); ax.set_xlabel('Escena'); ax.set_ylabel('Requests/batch')
    ax.tick_params(axis='x', rotation=30)

    plt.tight_layout(); plt.show()

    # --- 12c. Series temporales por escena ---
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk].copy()
        lbl = sub['label'].iloc[0]
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f'{lbl} ({bk})', fontsize=14)

        ax = axes[0,0]
        ax.plot(sub['batch_index'], sub['avg_fps'], marker='.', ms=3)
        ax.fill_between(sub['batch_index'], sub['min_fps'], sub['max_fps'], alpha=0.15)
        ax.set_title('FPS'); ax.set_xlabel('Batch'); ax.set_ylabel('FPS')

        ax = axes[0,1]
        ax.plot(sub['batch_index'], sub['avg_numVisible'], label='Visibles (frustum)', ms=3, marker='.')
        ax.plot(sub['batch_index'], sub['avg_numFiltered'], label='Post-occlusion', ms=3, marker='.')
        ax.set_title('Bloques: visibles vs post-occlusion'); ax.set_xlabel('Batch'); ax.legend()

        ax = axes[1,0]
        ax.plot(sub['batch_index'], sub['avg_activeCount'], color='green', ms=3, marker='.')
        ax.fill_between(sub['batch_index'], sub['min_activeCount'], sub['max_activeCount'], alpha=0.15, color='green')
        ax.set_title('\u00c1tomos rasterizados'); ax.set_xlabel('Batch'); ax.set_ylabel('\u00c1tomos')

        ax = axes[1,1]
        ax.plot(sub['batch_index'], sub['avg_numRequests'], color='orange', ms=3, marker='.')
        ax.fill_between(sub['batch_index'], sub['min_numRequests'], sub['max_numRequests'], alpha=0.15, color='orange')
        ax.set_title('Bloques solicitados a streaming'); ax.set_xlabel('Batch'); ax.set_ylabel('Requests')

        plt.tight_layout(); plt.show()

    # --- 12d. FPS vs \u00e1tomos activos (scatter, todas las escenas) ---
    fig, ax = plt.subplots(figsize=(12, 7))
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk]
        ax.scatter(sub['avg_activeCount'], sub['avg_fps'], s=15, alpha=0.5, label=bk)
    ax.set_xlabel('\u00c1tomos rasterizados promedio'); ax.set_ylabel('FPS promedio')
    ax.set_title('FPS vs \u00e1tomos rasterizados (todas las escenas)')
    ax.legend(title='Escena'); plt.tight_layout(); plt.show()

    # --- 12e. Estad\u00edstica descriptiva consolidada ---
    print('\n=== Estad\u00edstica descriptiva por escena ===')
    desc_cols = ['avg_frame_ms','avg_fps','avg_numVisible','avg_numFiltered','avg_numRequests','avg_activeCount']
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk]
        print(f'\n--- {bk} ({sub["label"].iloc[0]}) ---')
        display(sub[desc_cols].describe())


### 12f. Escalado: FPS medio vs cantidad de esferas (PACKAGE_SCENE)

Curva log-log de FPS medio por escena. Pendiente ideal: -1 (lineal). Incluye línea de tendencia.

In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    pkg_only = df_batch_all[df_batch_all['scene_type'].str.contains('PACKAGE', na=False)].copy()
    if not pkg_only.empty:
        sc_grp = pkg_only.groupby('escena_id').agg(
            sphere_count=('sphere_count','first'),
            fps_mean=('avg_fps','mean'), fps_std=('avg_fps','std'),
            fps_p25=('avg_fps', lambda x: x.quantile(0.25)),
            fps_p75=('avg_fps', lambda x: x.quantile(0.75)),
        ).reset_index().sort_values('sphere_count')

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        ax = axes[0]
        ax.errorbar(sc_grp['sphere_count']/1e6, sc_grp['fps_mean'],
                    yerr=sc_grp['fps_std'].fillna(0), marker='o', capsize=5, lw=2)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('FPS medio')
        ax.set_title('FPS vs Cantidad de entidades')
        ax.grid(True, which='both', alpha=0.3)
        if len(sc_grp) >= 2:
            coeffs = np.polyfit(np.log10(sc_grp['sphere_count']), np.log10(sc_grp['fps_mean']), 1)
            xs = np.logspace(np.log10(sc_grp['sphere_count'].min()), np.log10(sc_grp['sphere_count'].max()), 50)
            ax.plot(xs/1e6, 10**(coeffs[0]*np.log10(xs) + coeffs[1]), '--', color='red',
                    label=f'Tendencia: pendiente={coeffs[0]:.2f}')
            ax.legend()

        ax = axes[1]
        ax.errorbar(sc_grp['sphere_count']/1e6, sc_grp['fps_mean'],
                    yerr=[sc_grp['fps_mean']-sc_grp['fps_p25'], sc_grp['fps_p75']-sc_grp['fps_mean']],
                    marker='s', capsize=5, lw=2, color='darkorange')
        ax.set_xscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('FPS medio')
        ax.set_title('FPS con rango intercuart\u00edlico (P25-P75)')
        ax.grid(True, which='both', alpha=0.3)

        plt.tight_layout(); plt.show()


### 12g. Convergencia de streaming y eficacia del occlusion culling

**Requests vs tiempo:** cómo los bloques solicitados decaen a medida que el pool se llena.

**Ratio post-occlusion / visibles:** fracción de bloques que sobreviven al occlusion culling. Un valor de 0.2 indica que solo el 20% de los bloques visibles pasan la prueba de oclusión.

In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    n = len(order)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)
    fig.suptitle('Convergencia de streaming: requests por batch', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk].sort_values('batch_index')
        ax.semilogy(sub['batch_index'], sub['avg_numRequests'].clip(lower=0.1), marker='.', ms=3)
        ax.set_title(bk); ax.set_xlabel('Batch'); ax.set_ylabel('Requests (log)')
        ax.grid(True, alpha=0.3)
    for idx in range(n, rows*cols):
        r, c = divmod(idx, cols)
        axes[r][c].set_visible(False)
    plt.tight_layout(); plt.show()

    # Culling ratio
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)
    fig.suptitle('Ratio bloques post-occlusion / visibles (eficacia del occlusion culling)', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk].sort_values('batch_index')
        ratio = sub['avg_numFiltered'] / sub['avg_numVisible'].replace(0, np.nan)
        ax.plot(sub['batch_index'], ratio, marker='.', ms=3, color='teal')
        ax.set_title(bk); ax.set_xlabel('Batch'); ax.set_ylabel('Post-occlusion / Visibles')
        ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
    for idx in range(n, rows*cols):
        r, c = divmod(idx, cols)
        axes[r][c].set_visible(False)
    plt.tight_layout(); plt.show()


### 12h. Utilización del pool y fracción de átomos rasterizados

- **Pool utilization** = avg\_numVisible / pool\_slots: qué fracción de los slots del pool tienen bloques visibles.
- **Atom render fraction** = avg\_activeCount / sphere\_count: fracción de átomos rasterizados vs total de la escena.

In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Pool utilization boxplot
    df_batch_all['pool_util_pct'] = 100.0 * df_batch_all['avg_numVisible'] / df_batch_all['pool_slots'].replace(0, np.nan)
    ax = axes[0]
    sns.boxplot(data=df_batch_all, x='escena_id', y='pool_util_pct', order=order, ax=ax, palette='coolwarm')
    ax.set_title('Utilizaci\u00f3n del pool (bloques visibles / slots)'); ax.set_xlabel('Escena'); ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)

    # Active atom fraction
    df_batch_all['atom_fraction_pct'] = 100.0 * df_batch_all['avg_activeCount'] / df_batch_all['sphere_count'].replace(0, np.nan)
    ax = axes[1]
    sns.boxplot(data=df_batch_all, x='escena_id', y='atom_fraction_pct', order=order, ax=ax, palette='RdYlGn')
    ax.set_title('% \u00e1tomos rasterizados vs total esferas de la escena'); ax.set_xlabel('Escena'); ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)

    plt.tight_layout(); plt.show()

    # Summary table
    print('\n=== Pool utilization & atom density (medianas) ===')
    util_stats = df_batch_all.groupby('escena_id').agg(
        pool_util_median=('pool_util_pct','median'),
        pool_util_p95=('pool_util_pct', lambda x: x.quantile(0.95)),
        atom_frac_median=('atom_fraction_pct','median'),
        atom_frac_p95=('atom_fraction_pct', lambda x: x.quantile(0.95)),
    ).reindex(order)
    display(util_stats)


### 12i. Correlaciones entre métricas de batch (por escena)

Heatmaps de correlación Pearson entre frame\_ms, FPS, visibles, filtrados, requests y átomos.

In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    corr_cols = ['avg_frame_ms','avg_fps','avg_numVisible','avg_numFiltered','avg_numRequests','avg_activeCount']
    n = len(order)
    cols_per_row = min(3, n)
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(6*cols_per_row, 5*rows), squeeze=False)
    fig.suptitle('Correlaciones Pearson entre m\u00e9tricas de batch', fontsize=14)
    for idx, bk in enumerate(order):
        r, c = divmod(idx, cols_per_row)
        ax = axes[r][c]
        sub = df_batch_all[df_batch_all['escena_id'] == bk][corr_cols].dropna()
        if len(sub) >= 3:
            corr = sub.corr()
            sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                        vmin=-1, vmax=1, ax=ax, cbar=False,
                        xticklabels=[c.replace('avg_','') for c in corr_cols],
                        yticklabels=[c.replace('avg_','') for c in corr_cols])
        ax.set_title(bk)
    for idx in range(n, rows*cols_per_row):
        r, c = divmod(idx, cols_per_row)
        axes[r][c].set_visible(False)
    plt.tight_layout(); plt.show()


### 12j. Distribuciones de FPS y frame time por escena

Histogramas con KDE para entender la variabilidad en el rendimiento.

In [ ]:
if BATCH_CSVS and 'df_batch_all' in dir():
    order = sorted(df_batch_all['escena_id'].unique(), key=lambda x: _scene_sort_key(x))
    n = len(order)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 8), squeeze=False)
    fig.suptitle('Distribuciones de rendimiento por escena', fontsize=14)
    for idx, bk in enumerate(order):
        sub = df_batch_all[df_batch_all['escena_id'] == bk]
        ax = axes[0][idx]
        sub['avg_fps'].hist(ax=ax, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
        ax.set_title(f'{bk} - FPS'); ax.set_xlabel('FPS')
        if idx == 0: ax.set_ylabel('Frecuencia')

        ax = axes[1][idx]
        sub['avg_frame_ms'].hist(ax=ax, bins=40, alpha=0.7, color='coral', edgecolor='white')
        ax.set_title(f'{bk} - Frame time'); ax.set_xlabel('ms')
        if idx == 0: ax.set_ylabel('Frecuencia')

    plt.tight_layout(); plt.show()

    # Percentile table
    print('\n=== Percentiles de FPS por escena ===')
    pcts = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    pct_data = {}
    for bk in order:
        sub = df_batch_all[df_batch_all['escena_id'] == bk]['avg_fps']
        pct_data[bk] = sub.quantile(pcts).values
    pct_df = pd.DataFrame(pct_data, index=[f'P{int(p*100)}' for p in pcts])
    display(pct_df)


### 12k. Escalado del preprocesamiento: complejidad algorítmica

Log-log del número de bloques y nodos octree vs esferas. Escalado del block-file en disco.

In [ ]:
if PREPROCESS_CSV.exists() and 'prep_grp' in dir():
    pkg = prep_grp[prep_grp['scene_type'].str.contains('PACKAGE')].copy()
    if len(pkg) >= 2:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        ax = axes[0]
        ax.plot(pkg['sphere_count']/1e6, pkg['block_count'], marker='o', color='navy')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('Bloques')
        ax.set_title('Bloques generados vs entidades'); ax.grid(True, alpha=0.3)

        ax = axes[1]
        ax.plot(pkg['sphere_count']/1e6, pkg['octree_nodes'], marker='s', color='darkred')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('Nodos octree')
        ax.set_title('Nodos del octree vs entidades'); ax.grid(True, alpha=0.3)

        ax = axes[2]
        ax.plot(pkg['sphere_count']/1e6, pkg['avg_block_file_MB'], marker='^', color='darkgreen')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Millones de esferas'); ax.set_ylabel('Block file (MB)')
        ax.set_title('Tama\u00f1o del fichero de bloques'); ax.grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()


## 13. Comparativa inter-escena (si hay múltiples escenas cargadas)

In [ ]:
if len(dfs) >= 2:
    combined = []
    for scene_key, df in dfs.items():
        tmp = df.copy()
        tmp['scene'] = SCENES[scene_key]['label']
        combined.append(tmp)
    df_all = pd.concat(combined, ignore_index=True)

    for met in ['gr_active_pct', 'gr_idle_pct', 'l1tex_hit_pct', 'pcie_throughput']:
        if df_all[met].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(14, 6))
        sub = df_all[df_all['pipeline_range'].isin(focus_ranges)]
        sns.boxplot(data=sub, x='pipeline_range', y=met, hue='scene', ax=ax)
        ax.set_title(f'Comparativa — {met}')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('Solo una escena cargada, no se genera comparativa inter-escena.')

## 14. Memoria throughput — ¿Limitado por PCIe o por compute?

Si PCIe throughput domina (>60%) → el cuello de botella es la transferencia host-device (streaming).  
Si SM issue es alto y PCIe bajo → compute-bound.  
Si ambos son bajos → latency-bound (esperas, stalls).

In [ ]:
for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    sub = df[df['pipeline_range'].isin(focus_ranges)].copy()
    if sub.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 7))
    for rng in focus_ranges:
        rng_df = sub[sub['pipeline_range'] == rng]
        if rng_df[['pcie_throughput', 'sm_issue_active']].dropna().empty:
            continue
        ax.scatter(rng_df['pcie_throughput'], rng_df['sm_issue_active'],
                   label=rng, s=60, alpha=0.7)

    ax.axvline(60, ls='--', color='gray', alpha=0.5, label='PCIe 60%')
    ax.axhline(60, ls='--', color='gray', alpha=0.5, label='SM 60%')
    ax.set_xlabel('PCIe Throughput [%]')
    ax.set_ylabel('SM Issue Active [%]')
    ax.set_title(f"{cfg['label']} — ¿Memory-bound o Compute-bound?")
    ax.legend()
    ax.annotate('Latency-bound', xy=(5, 5), fontsize=11, color='red', alpha=0.6)
    ax.annotate('Compute-bound', xy=(5, 70), fontsize=11, color='blue', alpha=0.6)
    ax.annotate('Memory-bound', xy=(65, 5), fontsize=11, color='green', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 15. Resumen ejecutivo

In [ ]:
print('=' * 80)
print('RESUMEN EJECUTIVO — Análisis OOC Pipeline')
print('=' * 80)

for scene_key, df in dfs.items():
    cfg = SCENES[scene_key]
    print(f"\n{'─' * 60}")
    print(f"Escena: {cfg['label']}")
    print(f"Muestras: {df['sample'].nunique()} | Anillos: {df['ring'].nunique()}")

    grp = df.groupby('pipeline_range')

    # Top duration
    dur = grp['rel_frame_duration'].mean().sort_values(ascending=False)
    print(f"\nEtapas más costosas (fracción promedio del frame):")
    for rng, val in dur.head(5).items():
        print(f"  {rng:40s} {val:.4f} ({val*100:.1f}%)")

    # Idle
    idle = grp['gr_idle_pct'].mean().sort_values(ascending=False)
    high_idle = idle[idle > 50]
    if not high_idle.empty:
        print(f"\nEtapas con alto GPU idle (>50%):")
        for rng, val in high_idle.items():
            print(f"  {rng:40s} {val:.1f}%")

    # PCIe
    pcie = grp['pcie_throughput'].mean().sort_values(ascending=False)
    high_pcie = pcie[pcie > 30]
    if not high_pcie.empty:
        print(f"\nEtapas con alto PCIe throughput (>30%, posible bottleneck de memoria):")
        for rng, val in high_pcie.items():
            print(f"  {rng:40s} {val:.1f}%")

    # Cache
    cache = grp['l1tex_hit_pct'].mean().sort_values(ascending=True)
    low_cache = cache[cache < 60]
    if not low_cache.empty:
        print(f"\nEtapas con bajo L1TEX hit rate (<60%):")
        for rng, val in low_cache.items():
            print(f"  {rng:40s} {val:.1f}%")

if BATCH_CSVS and 'df_batch_all' in dir():
    print(f"\n{'─' * 60}")
    print("Stats Batch (runtime):")
    if 'avg_fps' in df_stats.columns:
        print(f"  FPS promedio:  {df_batch_all['avg_fps'].mean():.1f} (min {df_batch_all['avg_fps'].min():.1f}, max {df_batch_all['avg_fps'].max():.1f})")
    if 'avg_frame_ms' in df_stats.columns:
        print(f"  Frame time:    {df_batch_all['avg_frame_ms'].mean():.2f} ms")
    if 'avg_activeCount' in df_stats.columns:
        print(f"  Átomos activos promedio: {df_batch_all['avg_activeCount'].mean():.0f}")

if PREPROCESS_CSV.exists() and 'prep_grp' in dir():
    print(f"\n{'─' * 60}")
    print("Preprocesamiento:")
    if 'avg_ms' in prep_grp.columns:
        print(f"  Tiempo total: {prep_grp['avg_ms'].iloc[-1]:.1f} ms")
    if 'avg_throughput' in prep_grp.columns:
        print(f"  Throughput:   {prep_grp['avg_throughput'].iloc[-1]:.0f} átomos/ms")

print(f"\n{'=' * 80}")